In [1]:
!pip install fastapi uvicorn supabase python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.8 MB/s eta 0:00:00


In [2]:
from fastapi import FastAPI
import uvicorn

app = FastAPI()

@app.get("/")
def home():
    return {
        "status": "Mela al mando",
        "mensaje": "Backend de Andes City funcionando en la nube"
    }

print("Servidor configurado. Listo para despegar.")


Servidor configurado. Listo para despegar.


In [4]:
# Este código sirve para ejecutar el servidor dentro de Colab
import nest_asyncio
from pyngrok import ngrok

# Permitir que el servidor corra en este ambiente
nest_asyncio.apply()

# Iniciar el servidor de prueba
# Nota: En Colab no podemos usar el comando de terminal directo fácilmente,
# así que usamos esta línea de Python:
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

ModuleNotFoundError: No module named 'pyngrok'

In [5]:
!pip install pyngrok nest-asyncio

In [6]:
from pyngrok import ngrok

# Pega aquí el código que copiaste de la página de ngrok
conf_token = "3BdWu4PKaR73KpUBkw9PTlIxeDJ_2ngJLoEXW3jjrWY7SJK5y"
ngrok.set_auth_token(conf_token)

print("¡Túnel configurado con éxito, Mela!")

¡Túnel configurado con éxito, Mela!


In [7]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio
from pyngrok import ngrok

app = FastAPI()

# Esta es la ruta que te pidió el equipo (Endpoint de prueba)
@app.get("/")
def read_root():
    return {
        "status": "Mela al mando",
        "proyecto": "Andes City",
        "mensaje": "¡Backend funcionando desde la nube!"
    }

# Aplicar el parche para que funcione en Colab
nest_asyncio.apply()

# Abrir el túnel público
public_url = ngrok.connect(8000).public_url
print(f"🚀 TU BACKEND ESTÁ VIVO EN ESTA URL: {public_url}")


🚀 TU BACKEND ESTÁ VIVO EN ESTA URL: https://fleshlier-juxtapositional-latosha.ngrok-free.dev


In [8]:
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

In [9]:
import asyncio

# Configuración especial para que el servidor no choque con Colab
if __name__ == "__main__":
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)

    # En lugar de uvicorn.run, usamos este truco para Colab:
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

    print(f"✅ SERVIDOR ACTIVO EN SEGUNDO PLANO")
    print(f"🔗 Tu URL pública sigue siendo: {public_url}")

✅ SERVIDOR ACTIVO EN SEGUNDO PLANO
🔗 Tu URL pública sigue siendo: https://fleshlier-juxtapositional-latosha.ngrok-free.dev


In [11]:
# Traer todas las filas para ver qué agregó Dennys
try:
    check_weather = supabase.table("weather_log").select("*").execute()
    if check_weather.data:
        print("✅ ¡Datos encontrados en Clima!")
        print(f"Columnas detectadas: {list(check_weather.data[0].keys())}")
        print(f"Total de filas: {len(check_weather.data)}")
        # Mostrar la primera fila para revisar los datos
        print("Ejemplo de fila:", check_weather.data[0])
    else:
        print("⚠️ La tabla 'weather_log' existe pero está vacía.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")

✅ ¡Datos encontrados en Clima!
Columnas detectadas: ['id', 'temp_c', 'condition', 'alert_level', 'recorded_at']
Total de filas: 2
Ejemplo de fila: {'id': 1, 'temp_c': 10.5, 'condition': 'Lluvia Fuerte', 'alert_level': 'precaucion', 'recorded_at': '2026-03-29T21:54:43.400634+00:00'}


In [13]:
@app.get("/api/weather/current")
def get_weather():
    response = supabase.table("weather_log").select("*").order("recorded_at", desc=True).limit(1).execute()
    return response.data[0] if response.data else {"error": "Sin datos"}

@app.get("/api/routes/status")
def get_routes():
    # Cambiamos 'street_name' por 'name' que es como lo puso Dennys
    response = supabase.table("mobility_routes").select("name, status, travel_time_min").execute()
    return {"routes": response.data} if response.data else {"error": "Sin rutas"}

In [15]:
@app.get("/api/weather/current")
def get_weather():
    response = supabase.table("weather_log").select("*").order("recorded_at", desc=True).limit(1).execute()
    return response.data[0] if response.data else {"error": "Sin datos"}

@app.get("/api/routes/status")
def get_routes():
    # Cambiamos 'street_name' por 'name' que es como lo puso Dennys
    response = supabase.table("mobility_routes").select("name, status, travel_time_min").execute()
    return {"routes": response.data} if response.data else {"error": "Sin rutas"}

In [12]:
try:
    check_routes = supabase.table("mobility_routes").select("*").execute()
    if check_routes.data:
        print("✅ ¡Datos encontrados en Rutas!")
        print(f"Columnas detectadas: {list(check_routes.data[0].keys())}")
        print(f"Ejemplo de calle: {check_routes.data[0].get('street_name', 'N/A')}")
    else:
        print("⚠️ La tabla 'mobility_routes' está vacía.")
except Exception as e:
    print(f"❌ Error al conectar: {e}")


✅ ¡Datos encontrados en Rutas!
Columnas detectadas: ['id', 'name', 'status', 'affected_by_weather', 'travel_time_min', 'updated_at']
Ejemplo de calle: N/A


In [21]:
try:
    # Intentamos pedir una sola cosa a la tabla de clima
    supabase.table("weather_log").select("id").limit(1).execute()
    print("🟢 CONEXIÓN ACTIVA: El túnel hacia la base de datos está abierto.")
except Exception as e:
    print("🔴 ERROR DE CONEXIÓN: Revisa si las llaves o la URL son correctas.")

🟢 CONEXIÓN ACTIVA: El túnel hacia la base de datos está abierto.


In [22]:
from google.colab import userdata
from supabase import create_client, Client

# Ya no escribimos las llaves aquí, las traemos de los secretos de Colab
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')

# Conexión segura
supabase: Client = create_client(url, key)

print("✅ Conexión establecida de forma segura (Llaves ocultas)")

✅ Conexión establecida de forma segura (Llaves ocultas)
